### Practica HT1 - StageOps
**Modulo**: M4 - RA1
**Autores**: Ivan Trujillo, Daniel Esteban, Aitor Perez, Gabriel Herrador

In [1]:
import pandas as pd

df_reservations = pd.read_csv('../data/raw/reservations.csv')

display(df_reservations.head())

df_reservations.info()

,reservation_id,order_id,customer_id,concert_id,seat_id,created_at,expires_at,unit_price_eur
0,RES-0079057,ORD-0037789,CLI-041762,CON-03,S17-R27-B03,2026-10-16T10:00:10Z,2026-10-16T10:10:10Z,89.0
1,RES-0077493,ORD-0037037,CLI-056580,CON-03,S16-R07-B05,2026-10-16T10:17:58Z,2026-10-16T10:27:58Z,87.0
2,RES-0075550,ORD-0036125,CLI-079940,CON-03,S14-R12-B18,2026-10-16T11:30:44Z,2026-10-16T11:40:44Z,83.0
3,RES-0166308,ORD-0080055,CLI-073202,CON-06,S08-R11-B09,2026-10-23T10:15:42Z,2026-10-23T10:25:42Z,71.0
4,RES-0036832,ORD-0017597,CLI-069332,CON-02,S08-R19-B17,2026-10-03T10:01:22Z,2026-10-03T10:11:22Z,71.0


<class 'pandas.DataFrame'>
RangeIndex: 188788 entries, 0 to 188787
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   reservation_id  188788 non-null  str    
 1   order_id        188788 non-null  str    
 2   customer_id     188788 non-null  str    
 3   concert_id      188788 non-null  str    
 4   seat_id         188788 non-null  str    
 5   created_at      188788 non-null  str    
 6   expires_at      188788 non-null  str    
 7   unit_price_eur  188788 non-null  float64
dtypes: float64(1), str(7)
memory usage: 27.5 MB


In [6]:
import pandas as pd

# 1. Cargar el archivo tal cual (todo como texto/string por defecto)
df_reservations = pd.read_csv('../data/raw/reservations.csv', dtype=str)

print("--- 1. INFORMACIÓN GENERAL Y TOTAL DE FILAS ---")
print(f"Total de registros en reservations.csv: {len(df_reservations)}")
print(df_reservations.info())

print("\n--- 2. COMPROBACIÓN DE DUPLICADOS EN reservation_id ---")
duplicados_id = df_reservations['reservation_id'].duplicated().sum()
print(f"Número de reservation_id duplicados exactamente: {duplicados_id}")

print("\n--- 3. INSPECCIÓN DE FORMATOS DE FECHA EN TEXTO (`created_at`) ---")
# Ver la longitud de las cadenas de fecha para detectar posibles anomalías o formatos truncados
longitudes_fechas = df_reservations['created_at'].str.len().value_counts()
print("Distribución de longitudes en las cadenas de created_at:")
print(longitudes_fechas)

print("\n--- 4. ANÁLISIS DE PEDIDOS (`order_id`) ---")
# Ver cuántas reservas (butacas) suele agrupar un mismo order_id de forma general
conteo_por_pedido = df_reservations['order_id'].value_counts()
print("Top 5 pedidos con más butacas asociadas:")
print(conteo_por_pedido.head())
print(f"Máximo de butacas en un solo pedido: {conteo_por_pedido.max()}")



--- 1. INFORMACIÓN GENERAL Y TOTAL DE FILAS ---
Total de registros en reservations.csv: 188788
<class 'pandas.DataFrame'>
RangeIndex: 188788 entries, 0 to 188787
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   reservation_id  188788 non-null  str  
 1   order_id        188788 non-null  str  
 2   customer_id     188788 non-null  str  
 3   concert_id      188788 non-null  str  
 4   seat_id         188788 non-null  str  
 5   created_at      188788 non-null  str  
 6   expires_at      188788 non-null  str  
 7   unit_price_eur  188788 non-null  str  
dtypes: str(8)
memory usage: 28.5 MB
None

--- 2. COMPROBACIÓN DE DUPLICADOS EN reservation_id ---
Número de reservation_id duplicados exactamente: 0

--- 3. INSPECCIÓN DE FORMATOS DE FECHA EN TEXTO (`created_at`) ---
Distribución de longitudes en las cadenas de created_at:
created_at
20    188788
Name: count, dtype: int64

--- 4. ANÁLISIS DE PEDIDOS (`order_id`) 

In [7]:
vacios_o_espacios = {}
for col in df_reservations.columns:
    # Contar cuántas celdas están vacías tras quitar espacios en blanco
    n_vacios = (df_reservations[col].str.strip() == "").sum()
    vacios_o_espacios[col] = n_vacios

print("--- RECUENTO DE VALORES VACÍOS O EN BLANCO ---")
for col, count in vacios_o_espacios.items():
    print(f"- {col}: {count} valores vacíos")

--- RECUENTO DE VALORES VACÍOS O EN BLANCO ---
- reservation_id: 0 valores vacíos
- order_id: 0 valores vacíos
- customer_id: 0 valores vacíos
- concert_id: 0 valores vacíos
- seat_id: 0 valores vacíos
- created_at: 0 valores vacíos
- expires_at: 0 valores vacíos
- unit_price_eur: 0 valores vacíos


In [8]:
import pandas as pd

# Cargar los datasets necesarios para la auditoría de cruces (en modo texto/string)
df_reservations = pd.read_csv('../data/raw/reservations.csv', dtype=str)
df_concerts = pd.read_csv('../data/raw/concerts.csv', dtype=str)
df_customers = pd.read_csv('../data/raw/customers.csv', dtype=str)
df_payments = pd.read_csv('../data/raw/payment_events.csv', dtype=str)

print("--- 1. AUDITORÍA DE CONCIERTOS (`concert_id`) ---")
# Comprobar si hay concert_ids en reservations que no existan en concerts
concerts_unicos_res = set(df_reservations['concert_id'])
concerts_unicos_cat = set(df_concerts['concert_id'])
huerfanos_concert = concerts_unicos_res - concerts_unicos_cat
print(concerts_unicos_res)
print(concerts_unicos_cat)
print(f"Conciertos en reservas que no figuran en concerts.csv: {len(huerfanos_concert)}")

print("\n--- 2. AUDITORÍA DE CLIENTES (`customer_id`) ---")
customers_unicos_res = set(df_reservations['customer_id'])
customers_unicos_cat = set(df_customers['customer_id'])
huerfanos_customer = customers_unicos_res - customers_unicos_cat
print(f"Clientes en reservas que no figuran en customers.csv: {len(huerfanos_customer)}")

print("\n--- 3. AUDITORÍA DE PAGOS (`reservation_id`) ---")
# Ver si hay reservas que no tienen evento de pago o si hay pagos de reservas que no existen
res_ids = set(df_reservations['reservation_id'])
pay_res_ids = set(df_payments['reservation_id'])

pagos_sin_reserva = pay_res_ids - res_ids
print(f"Pagos registrados cuya reservation_id no existe en reservas: {len(pagos_sin_reserva)}")

--- 1. AUDITORÍA DE CONCIERTOS (`concert_id`) ---
{'CON-02', 'CON-06', 'CON-01', 'CON-05', 'CON-04', 'CON-03'}
{'CON-02', 'CON-06', 'CON-01', 'CON-05', 'CON-04', 'CON-03'}
Conciertos en reservas que no figuran en concerts.csv: 0

--- 2. AUDITORÍA DE CLIENTES (`customer_id`) ---
Clientes en reservas que no figuran en customers.csv: 0

--- 3. AUDITORÍA DE PAGOS (`reservation_id`) ---
Pagos registrados cuya reservation_id no existe en reservas: 0


In [9]:
import pandas as pd

# Cargar el archivo tal cual (en texto)
df_reservations = pd.read_csv('../data/raw/reservations.csv', dtype=str)

# Convertir temporalmente en memoria a datetime solo para calcular la diferencia
created_dt = pd.to_datetime(df_reservations['created_at'])
expires_dt = pd.to_datetime(df_reservations['expires_at'])

# Calcular la diferencia en minutos
diferencia_minutos = (expires_dt - created_dt).dt.total_seconds() / 60

# Ver los diferentes tiempos de bloqueo encontrados en todo el dataset
tiempos_unicos = diferencia_minutos.unique()
print("--- COMPROBACIÓN DE LA REGLA DE BLOQUEO (10 MINUTOS) ---")
print(f"Duraciones encontradas (minutos): {tiempos_unicos}")

# Comprobar si hay alguna reserva que no dure exactamente 10 minutos
anomalas = df_reservations[diferencia_minutos != 10.0]
print(f"Número de reservas que NO cumplen exactamente los 10 minutos: {len(anomalas)}")

if len(anomalas) > 0:
    print("\n⚠️ Primeras reservas con anomalías en el tiempo de bloqueo:")
    display(anomalas[['reservation_id', 'concert_id', 'seat_id', 'created_at', 'expires_at']].head())
else:
    print("\n✅ ¡Todas las reservas del dataset cumplen estrictamente con la ventana exacta de 10 minutos!")

--- COMPROBACIÓN DE LA REGLA DE BLOQUEO (10 MINUTOS) ---
Duraciones encontradas (minutos): [10.]
Número de reservas que NO cumplen exactamente los 10 minutos: 0

✅ ¡Todas las reservas del dataset cumplen estrictamente con la ventana exacta de 10 minutos!


In [11]:
df_reservations['created_at'] = pd.to_datetime(df_reservations['created_at'])
df_reservations['expires_at'] = pd.to_datetime(df_reservations['expires_at'])
print("--- INFO ACTUALIZADO CON LOS TIPOS CORRECTOS ---")
print(df_reservations.info())

# Mostrar las primeras filas de la tabla para verificar visualmente el cambio
print("\n--- VISTA PREVIA DE LA TABLA ---")
display(df_reservations.head())

--- INFO ACTUALIZADO CON LOS TIPOS CORRECTOS ---
<class 'pandas.DataFrame'>
RangeIndex: 188788 entries, 0 to 188787
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   reservation_id  188788 non-null  str                
 1   order_id        188788 non-null  str                
 2   customer_id     188788 non-null  str                
 3   concert_id      188788 non-null  str                
 4   seat_id         188788 non-null  str                
 5   created_at      188788 non-null  datetime64[us, UTC]
 6   expires_at      188788 non-null  datetime64[us, UTC]
 7   unit_price_eur  188788 non-null  str                
dtypes: datetime64[us, UTC](2), str(6)
memory usage: 21.3 MB
None

--- VISTA PREVIA DE LA TABLA ---


,reservation_id,order_id,customer_id,concert_id,seat_id,created_at,expires_at,unit_price_eur
0,RES-0079057,ORD-0037789,CLI-041762,CON-03,S17-R27-B03,2026-10-16 10:00:10+00:00,2026-10-16 10:10:10+00:00,89.00
1,RES-0077493,ORD-0037037,CLI-056580,CON-03,S16-R07-B05,2026-10-16 10:17:58+00:00,2026-10-16 10:27:58+00:00,87.00
2,RES-0075550,ORD-0036125,CLI-079940,CON-03,S14-R12-B18,2026-10-16 11:30:44+00:00,2026-10-16 11:40:44+00:00,83.00
3,RES-0166308,ORD-0080055,CLI-073202,CON-06,S08-R11-B09,2026-10-23 10:15:42+00:00,2026-10-23 10:25:42+00:00,71.00
4,RES-0036832,ORD-0017597,CLI-069332,CON-02,S08-R19-B17,2026-10-03 10:01:22+00:00,2026-10-03 10:11:22+00:00,71.00


In [1]:
import pandas as pd

# 1. Cargar el dataset raw preservando los tipos originales
df_reservations = pd.read_csv('../data/raw/reservations.csv', dtype=str)

# 2. Convertir las columnas de fecha al formato datetime con zona horaria UTC
df_reservations['created_at'] = pd.to_datetime(df_reservations['created_at'])
df_reservations['expires_at'] = pd.to_datetime(df_reservations['expires_at'])

# 3. Exportar el DataFrame limpio a un nuevo archivo CSV
output_path = 'reservations_clean.csv'
df_reservations.to_csv(output_path, index=False)

print(f"Archivo generado con éxito: {output_path}")

Archivo generado con éxito: reservations_clean.csv
